#Practice of User-Defined Functions (UDF) for different use cases

In [0]:
%sql
SELECT current_catalog(), current_schema();


In [0]:
%sql
SHOW TABLES;


In [0]:
%sql
-- User-Defined Function examples:

-- Mask a string value except for the 2-left most characters
CREATE OR REPLACE FUNCTION mask_string(value STRING)
RETURNS STRING
RETURN CONCAT(SUBSTRING(value, 1, 2), REPEAT('*', LENGTH(value) - 2));

-- Mask a string value except for the last 4 characters
CREATE OR REPLACE FUNCTION mask_string_last(value STRING)
--RETURNS STRING
RETURN CONCAT(REPEAT('*', LENGTH(value) - 4), SUBSTRING(value, -4));

In [0]:
%sql
SELECT mask_string("Hola mundo!");

In [0]:
%sql
SELECT mask_string_last('1234567890') AS mask_string_last;

In [0]:
%sql
-- Delete an UDF
drop function if exists mask_string;

##UDF for Row Filtering

In [0]:
%sql
-- Let´s just view the table to play with
select * from people_table;

In [0]:
%sql
-- Step 1: Funtion to filter names starting with a vowel, namely true if Alice, Eva, Ian for non-Admin users
create function name_filter (name string)
return if(is_member('admin'), true, upper(substring(name, 1, 1) in ('A', 'E', 'I', 'O', 'U')));

In [0]:
%sql
-- Step 2:
alter table people_table set row filter name_filter on (name);

In [0]:
%sql
select * from people_table where name_filter(name);

In [0]:
%sql
select * from people_table;

In [0]:
%sql
-- Revoke the filter
alter table people_table drop row filter;

##UDF for Mask Column 

In [0]:
%sql
-- Step 1: Function to mask the column salary for all non-Admin users
create function salary_mask (salary int)
return if(is_member('admin'), salary, 0);

In [0]:
%sql
--Step 2: Alter table to apply the mask
alter table people_table alter column salary set mask salary_mask;

In [0]:
%sql
select * from people_table;

In [0]:
%sql
-- Revoke the mask
alter table people_table alter column salary drop mask;

In [0]:
%sql
-- Check if the table has a mask
show create table people_table; -- output should not have references to any mask

In [0]:
%sql
select current_user();

In [0]:
%sql
grant all privileges on table people_table to `almayo@gmail.com`;

In [0]:
%sql
select * from people_table;

##Let´s do certain filtering on cell level. Just for fun

In [0]:
%sql
CREATE OR REPLACE FUNCTION mask_string(value STRING)
RETURNS STRING
RETURN CONCAT(SUBSTRING(value, 1, 2), REPEAT('*', LENGTH(value) - 2));

In [0]:
%sql
-- Task: Do a query to show name column if admin role and call mask_string function for rest of users. Also filter out the rows with salary below 60000 
select 
  id,
  case when is_member('admin') then name else mask_string(name) end as name
from people_table
where salary > 60000;